# 365 Probabilidades - Dia #114
## Qual a probabilidade de uma prefeita ser reeleita?

**Tipo:** Institucional
**Data de publicação:** 2026-10-05
**Ferramenta:** Python
**Decisão analisada:** Por que o ambiente continua dominado por homens?
**Hashtag:** #365Probabilidades #Dia114

---

### 📖 A História

Eu tenho olhado cada vez mais para essa parte.

Quantas mulheres estão na sala. Quantas aparecem na lista. Quantas ficam depois que a
lista vira resultado.

E quando os resultados saem, o que eu vejo continua sendo um ambiente dominado por
homens.

A pergunta que eu queria responder não é se isso é verdade. É de onde isso vem.

---

### 📚 O Conceito

Comparar prefeitos e prefeitas direto não serve para nada. Cidades que elegem mulheres
podem ser diferentes de cidades que elegem homens em dezenas de coisas, e aí não dá
para saber o que é efeito de quem governa.

A saída é olhar só as **corridas apertadas**. Quando um homem e uma mulher terminam
separados por uma fração de ponto percentual, quem ganhou é quase sorteio. As cidades
dos dois lados do empate são parecidas em tudo, menos no gênero de quem assumiu.

É isso que uma descontinuidade de regressão faz: compara os dois lados de um resultado
quase empatado, e trata a diferença como efeito.

---

### 🧮 O Modelo

O artigo mede o que acontece dentro do mandato e na hora da reeleição.

O que eu faço aqui é diferente: pego a penalidade de reeleição medida por eles e
pergunto **quanto da dominância masculina ela consegue explicar sozinha**.

Para isso modelo a renovação dos cargos ao longo de vários ciclos, respeitando a regra
brasileira de que prefeito só pode disputar **uma** reeleição consecutiva.

A fração de mulheres no estado estável tem forma fechada:

$$s^{*} = \frac{w\,(1 + r_f)}{1 + w\,r_f + (1-w)\,r_m}$$

onde **w** é a fração de mulheres entre quem entra num cargo aberto, **r_m** e **r_f**
são as probabilidades de reeleição de homens e mulheres, e **r_f = r_m − 0,20**.

**Fontes:**
- Brollo, F. & Troiano, U. (2016). *What happens when a woman wins an election?
  Evidence from close races in Brazil.* Journal of Development Economics 122, 28-45.
  doi:10.1016/j.jdeveco.2016.04.003
- Tamanho da amostra (723 corridas apertadas, eleições de 2000 e 2004) conforme a
  versão de working paper: CAGE Working Paper 161 / MPRA 52244 (2013)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

print("✅ Bibliotecas carregadas")

In [ ]:
# --- O QUE O ARTIGO MEDE ---
# Brollo & Troiano (2016), Journal of Development Economics 122, 28-45.
# Descontinuidade de regressao em corridas apertadas entre um homem e uma mulher.
# Corrupcao medida por auditorias sorteadas da CGU, nao por autorrelato.

N_CORRIDAS = 723        # versao de working paper (CAGE WP 161, 2013)
ELEICOES = (2000, 2004)

# Achados do resumo publicado. Sao as estimativas DELES, nao minhas.
CORRUPCAO_MENOR = (0.29, 0.35)   # 29% a 35% menos episodios de corrupcao
TRANSFERENCIAS_MAIS = 0.60       # ~60% mais transferencias para investimento
PENALIDADE = 0.20                # ~20 PONTOS PERCENTUAIS a menos de reeleicao
# O artigo tambem reporta, sem quantificar no resumo:
#   - prefeitas contratam menos temporarios no ano eleitoral
#   - prefeitas que tentam a reeleicao arrecadam menos doacao de campanha

print("=" * 70)
print("  O QUE O ARTIGO ENCONTRA")
print("=" * 70)
print(f"\n  Desenho:   descontinuidade em {N_CORRIDAS} corridas apertadas")
print(f"  Eleições:  municipais de {ELEICOES[0]} e {ELEICOES[1]}")
print(f"  Corrupção: medida por auditoria sorteada da CGU\n")
print(f"  Comparando prefeitas com prefeitos:")
print(f"    episódios de corrupção     {CORRUPCAO_MENOR[0]*100:.0f}% a "
      f"{CORRUPCAO_MENOR[1]*100:.0f}% MENOS")
print(f"    transferências de investimento   ~{TRANSFERENCIAS_MAIS*100:.0f}% MAIS")
print(f"    contratação de temporários       MENOS (não quantificado no resumo)")
print(f"    doação de campanha na reeleição  MENOS (não quantificado no resumo)")
print(f"    probabilidade de reeleição       {PENALIDADE*100:.0f} PONTOS a menos")
print(f"\n  Governam mais limpo por essas medidas, e voltam menos.")
print(f"\n  ×0,80: não se aplica. Auditoria e registro eleitoral, não autorrelato.")
print("=" * 70)

In [ ]:
# --- O MODELO ---
# Pergunta: quanto da dominancia masculina a penalidade de reeleicao explica sozinha?
#
# Modelo de renovacao do cargo, com a regra brasileira de UMA reeleicao consecutiva.
# Estados: prefeito de 1o mandato (pode disputar de novo) e de 2o mandato (nao pode).
#   w  = fracao de mulheres entre quem entra num cargo aberto
#   rm = probabilidade de reeleicao dos homens
#   rf = rm - PENALIDADE

def estavel(w, rm, gap=None):
    # Fracao de mulheres no estado estavel, com limite de dois mandatos.
    gap = PENALIDADE if gap is None else gap
    rf = rm - gap
    return w * (1 + rf) / (1 + w * rf + (1 - w) * rm)

def estavel_sem_limite(w, rm, gap=None):
    # Mesmo modelo sem o limite de mandatos. Serve so de comparacao.
    gap = PENALIDADE if gap is None else gap
    rf = rm - gap
    return w * (1 - rm) / (1 - rf + w * (rf - rm))

def por_iteracao(w, rm, gap=None, ciclos=600):
    # Confere a formula rodando o processo ciclo a ciclo.
    gap = PENALIDADE if gap is None else gap
    rf = rm - gap
    f1 = f2 = w / 2
    m1 = m2 = (1 - w) / 2
    for _ in range(ciclos):
        nf2, nm2 = f1 * rf, m1 * rm
        aberto = 1 - nf2 - nm2
        f1, m1 = w * aberto, (1 - w) * aberto
        f2, m2 = nf2, nm2
    return f1 + f2

print("=" * 72)
print("  CAMADA 1 · A FÓRMULA CONFERE")
print("=" * 72)
print(f"\n  fórmula fechada: {estavel(0.50, 0.45):.6f}")
print(f"  iteração direta: {por_iteracao(0.50, 0.45):.6f}")
assert abs(estavel(0.50, 0.45) - por_iteracao(0.50, 0.45)) < 1e-9
print("  conferem.")

# ── CAMADA 2 · o caso limpo: entrada perfeitamente paritaria ──
print("\n" + "=" * 72)
print("  CAMADA 2 · SE A ENTRADA FOSSE METADE E METADE")
print("=" * 72)
print(f"\n  Cada cargo aberto sendo disputado 50/50, e nada mais mudando.\n")
print(f"  {'reeleição homens':>18}{'mulheres':>11}{'estado estável':>17}{'déficit':>11}")
RMS = [0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
defs = []
for rm in RMS:
    s = estavel(0.50, rm)
    defs.append((50 - s * 100))
    print(f"  {rm*100:>16.0f}%{(rm-PENALIDADE)*100:>10.0f}%{s*100:>16.1f}%"
          f"{50-s*100:>10.1f}pp")
print(f"\n  O déficit fica entre {vir(min(defs))} e {vir(max(defs))} pontos,")
print(f"  seja qual for a taxa de reeleição dos homens que eu use.")
print(f"  Não é 50%. É perto de {vir(estavel(0.50, 0.45)*100)}%.")

# ── CAMADA 3 · e se a entrada nao for paritaria ──
print("\n" + "=" * 72)
print("  CAMADA 3 · PARA QUALQUER PONTO DE PARTIDA")
print("=" * 72)
print(f"\n  {'entram':>10}{'terminam':>12}{'encolhimento':>16}")
razoes = []
for w in [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
    s = estavel(w, 0.45)
    razoes.append(s / w)
    print(f"  {w*100:>9.0f}%{s*100:>11.1f}%{(1-s/w)*100:>14.1f}%")
print(f"\n  A penalidade encolhe a representação em {vir((1-max(razoes))*100)}% a "
      f"{vir((1-min(razoes))*100)}%,")
print(f"  seja qual for a fração de mulheres que entra.")

# ── CAMADA 4 · quanto disso e a penalidade, e quanto e a regra ──
print("\n" + "=" * 72)
print("  CAMADA 4 · O TAMANHO DA PENALIDADE, E O PAPEL DO LIMITE DE MANDATOS")
print("=" * 72)
print(f"\n  Entrada paritária, reeleição dos homens em 45%.\n")
print(f"  {'penalidade':>12}{'com limite':>14}{'sem limite':>14}")
for g in [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
    a, b = estavel(0.50, 0.45, g), estavel_sem_limite(0.50, 0.45, g)
    print(f"  {g*100:>10.0f}pp{50-a*100:>12.1f}pp{50-b*100:>13.1f}pp")
print(f"\n  A regra de uma reeleição só corta o efeito quase pela metade.")
print(f"  Um modelo que ignorasse o limite de mandatos exageraria o déficit.")

# ── CAMADA 5 · o que sobra ──
s45 = estavel(0.50, 0.45)
print("\n" + "=" * 72)
print("  CAMADA 5 · O QUE A PENALIDADE NÃO EXPLICA")
print("=" * 72)
print(f"\n  Com entrada paritária, a penalidade sozinha leva de 50% para "
      f"{vir(s45*100)}%.")
print(f"  São {vir(50-s45*100)} pontos.")
print(f"\n  O ambiente real não está em {vir(s45*100)}%. Está muito abaixo disso.")
print(f"  Logo, a maior parte da diferença não está na reeleição.")
print(f"  Está antes: em quem entra na disputa.")
print("=" * 72)

In [ ]:
# --- VISUALIZAÇÃO ---

# ══ GRÁFICO 1 · o que muda na gestão ══
fig1, ax1 = plt.subplots(figsize=(11, 5.9))
rot = ['episódios de\ncorrupção', 'transferências\npara investimento']
c_lo, c_hi = (1 - CORRUPCAO_MENOR[1]) * 100, (1 - CORRUPCAO_MENOR[0]) * 100
c_mid = (c_lo + c_hi) / 2
t_val = (1 + TRANSFERENCIAS_MAIS) * 100

ax1.barh(1, c_mid - 100, left=100, color=VERDE, alpha=0.9, height=0.32)
ax1.errorbar(c_mid, 1, xerr=[[c_mid - c_lo], [c_hi - c_mid]], fmt='none',
             ecolor='#123f3b', elinewidth=2.4, capsize=9)
ax1.barh(0, t_val - 100, left=100, color=DOURADO, alpha=0.9, height=0.32)

ax1.axvline(100, color='#333333', lw=2)
ax1.text(100, 1.60, 'prefeito homem = 100', ha='center', fontsize=11.5, color='#333333')
ax1.text(c_lo - 3.5, 1, '65 a 71', va='center', ha='right', fontsize=16,
         fontweight='bold', color=VERDE)
ax1.text(t_val + 3.5, 0, '160', va='center', ha='left', fontsize=16,
         fontweight='bold', color='#8a6f1f')
ax1.set_yticks([1, 0])
ax1.set_yticklabels(rot, fontsize=12.5)
ax1.set_xlim(45, 185)
ax1.set_ylim(-0.55, 1.9)
ax1.set_xlabel('índice, com o prefeito homem em 100', fontsize=11)
ax1.set_title('O que muda quando uma mulher ganha a eleição\n'
              f'{N_CORRIDAS} corridas apertadas, municipais de 2000 e 2004',
              fontsize=14, pad=16)
plt.figtext(0.5, 0.005, 'Fonte: Brollo & Troiano (2016), Journal of Development '
            'Economics · corrupção medida por auditoria sorteada da CGU · a penalidade '
            'de reeleição\nestá no gráfico 2, em pontos percentuais, que é a unidade '
            'em que o artigo a reporta  |  #365Probabilidades',
            ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=[0, 0.065, 1, 1])
plt.savefig('dia-114-grafico-01-gestao.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# ══ GRÁFICO 2 · a penalidade e o que ela produz ══
fig2, (ax2a, ax2b) = plt.subplots(1, 2, figsize=(13.5, 6))

rms = np.linspace(0.30, 0.65, 200)
ax2a.plot(rms * 100, rms * 100, color=CINZA, lw=3, label='prefeitos')
ax2a.plot(rms * 100, (rms - PENALIDADE) * 100, color=VERMELHO, lw=3,
          label='prefeitas')
ax2a.fill_between(rms * 100, (rms - PENALIDADE) * 100, rms * 100,
                  color=VERMELHO, alpha=0.10)
ax2a.text(56, 42, f'{PENALIDADE*100:.0f} pontos', fontsize=13,
          color=VERMELHO, fontweight='bold', ha='center')
ax2a.annotate('', xy=(52, 52), xytext=(52, 32),
              arrowprops=dict(arrowstyle='<->', color=VERMELHO, lw=1.8))
ax2a.set_xlabel('taxa de reeleição dos prefeitos (%)', fontsize=11)
ax2a.set_ylabel('probabilidade de reeleição (%)', fontsize=11)
ax2a.set_xlim(30, 65); ax2a.set_ylim(5, 70)
ax2a.legend(fontsize=11, loc='upper left')
ax2a.set_title('A penalidade medida pelo artigo', fontsize=13, pad=12)

ws = np.linspace(0.02, 0.60, 300)
ax2b.plot(ws * 100, ws * 100, color=CINZA, lw=3, ls='--',
          label='sem penalidade')
ax2b.plot(ws * 100, [estavel(w, 0.45) * 100 for w in ws], color=VERMELHO, lw=3,
          label='com a penalidade de 20 pp')
ax2b.fill_between(ws * 100, [estavel(w, 0.45) * 100 for w in ws], ws * 100,
                  color=VERMELHO, alpha=0.10)
s50 = estavel(0.50, 0.45) * 100
ax2b.plot([50], [s50], 'o', color=VERMELHO, ms=10, zorder=5)
ax2b.plot([50], [50], 'o', color=CINZA, ms=9, zorder=5)
ax2b.annotate(f'entrada paritária\ntermina em {vir(s50)}%',
              xy=(50, s50), xytext=(28, 52), fontsize=11.5, color=VERMELHO,
              fontweight='bold',
              arrowprops=dict(arrowstyle='->', color=VERMELHO, lw=1.6))
ax2b.set_xlabel('mulheres entre quem entra num cargo aberto (%)', fontsize=11)
ax2b.set_ylabel('mulheres no estado estável (%)', fontsize=11)
ax2b.text(3, 56, 'a distância entre as duas linhas\né o efeito inteiro',
          fontsize=10.5, color=CINZA, style='italic')
ax2b.set_xlim(0, 60); ax2b.set_ylim(0, 62)
ax2b.legend(fontsize=11, loc='lower right')
ax2b.set_title('O que ela produz ao longo dos ciclos', fontsize=13, pad=12)

plt.figtext(0.5, 0.005, 'Modelo de renovação do cargo com a regra brasileira de uma '
            'reeleição consecutiva · penalidade de Brollo & Troiano (2016)'
            '  |  #365Probabilidades', ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-114-grafico-02-modelo.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# ══ GRÁFICO 3 · o tamanho do efeito, e o papel da regra ══
fig3, ax3 = plt.subplots(figsize=(11, 6.2))
gs = np.linspace(0, 0.32, 300)
com = [50 - estavel(0.50, 0.45, g) * 100 for g in gs]
sem = [50 - estavel_sem_limite(0.50, 0.45, g) * 100 for g in gs]
ax3.plot(gs * 100, sem, color=CINZA, lw=2.6, ls='--',
         label='modelo sem o limite de mandatos')
ax3.plot(gs * 100, com, color=VERMELHO, lw=3.2,
         label='modelo com uma reeleição só, que é a regra')
d20 = 50 - estavel(0.50, 0.45) * 100
ax3.plot([20], [d20], 'o', color=VERMELHO, ms=11, zorder=5)
ax3.axvline(20, color='#bbbbbb', lw=1, ls=':')
ax3.annotate(f'a penalidade medida:\n{vir(d20)} pontos de déficit',
             xy=(20, d20), xytext=(22.5, 2.0), fontsize=12, color=VERMELHO,
             fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=VERMELHO, lw=1.6))
ax3.set_xlabel('tamanho da penalidade de reeleição (pontos percentuais)', fontsize=11)
ax3.set_ylabel('déficit de mulheres no estado estável (pp)', fontsize=11)
ax3.set_xlim(0, 32); ax3.set_ylim(0, 12)
ax3.legend(fontsize=11, loc='upper left')
ax3.set_title('Quanto a penalidade explica, e por que a regra importa\n'
              'partindo de entrada paritária nos cargos abertos',
              fontsize=13.5, pad=16)
plt.figtext(0.5, 0.005, 'Um modelo que ignorasse o limite de dois mandatos '
            'exageraria o déficit em quase o dobro  |  #365Probabilidades',
            ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=[0, 0.035, 1, 1])
plt.savefig('dia-114-grafico-03-sensibilidade.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

### 💡 O Insight

Quando uma mulher ganha uma eleição apertada no Brasil, a cidade dela registra **29% a
35% menos episódios de corrupção** nas auditorias sorteadas da CGU, e recebe **cerca de
60% mais transferências para investimento**.

E ela tem **20 pontos percentuais a menos** de chance de ser reeleita.

O artigo aponta por onde isso passa: prefeitas que tentam a reeleição **arrecadam menos
doação de campanha**. O mecanismo é financeiro, não moral.

Então eu peguei essa penalidade e perguntei o quanto dela explica o que eu vejo quando
olho os resultados.

Se a entrada fosse perfeitamente paritária, metade dos cargos abertos disputados por
mulheres, o estado estável não seria 50%. Seria **46,3%**.

São **3,7 pontos**, e esse número é firme: fica entre 3,3 e 4,0 para qualquer taxa de
reeleição masculina plausível.

Quase errei esse número por um fator de dois. A primeira versão do modelo ignorava que
prefeito no Brasil só pode disputar **uma** reeleição consecutiva. Sem essa regra, o
déficit dava 7,7 pontos. Com ela, 3,7. A regra corta quase pela metade, porque limita a
acumulação de qualquer vantagem de incumbência, inclusive a deles.

E aí vem a parte que eu não esperava.

**3,7 pontos é pouco.**

O ambiente que eu vejo quando olho os resultados não está em 46%. Está muito, muito
abaixo. Se a penalidade de reeleição explicasse tudo, bastaria consertar a reeleição.
Não basta.

A penalidade é real, foi medida com auditoria e não com opinião, e é injusta. Mas ela é
a parte pequena.

O grosso acontece antes de qualquer urna: em quem se candidata.

*A conta que eu fiz mede quem sai. Quantas você conhece que nunca entraram?*

---

### ⚠️ Limitações do Modelo

- Os 20 pontos vêm de uma descontinuidade em corridas apertadas. É o efeito **perto do
  empate**, não a média de todas as prefeitas do Brasil. Meu modelo pergunta o que
  aconteceria se valesse para todas, e isso é premissa minha, não achado deles.
- Os coeficientes são do artigo. Não tenho o microdado, então não refiz a
  descontinuidade nem recalculei nenhum intervalo. O que é meu aqui é o modelo de
  renovação, não a estimativa.
- Os dados são de mandatos municipais de 2001 a 2008. Nada aqui é sobre a eleição de
  ontem, nem sobre qualquer candidatura específica.
- O modelo supõe que a fração de mulheres entre os entrantes não muda ao longo dos
  ciclos. Se ver uma prefeita faz mais mulheres se candidatarem, o déficit real é menor
  que o meu.
- O modelo trata reeleição como o único caminho de permanência. Ignora quem sai e volta
  depois, e quem muda de cargo.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
